# Medallion Architecture — 00 Basic

Goal: understand the concept first.

Medallion is just a simple data flow:

```text
RAW DATA
   |
   v
BRONZE  = keep raw data
   |
   v
SILVER  = clean valid data
   |           |         -> QUARANTINE = bad rows
   v
GOLD    = business result
```

In this notebook we keep the example intentionally small.

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("medallion-basic")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_medallion_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Raw input

Small intentionally dirty dataset.

```text
+----------+---------+--------+
| event_id | user_id | amount |
+----------+---------+--------+
| e1       | u1      | 10.0   | valid
| e2       | null    | 20.0   | invalid: missing user
| e3       | u2      | -5.0   | invalid: negative amount
| e1       | u1      | 10.0   | duplicate event
+----------+---------+--------+
```

In [2]:
raw_rows = [
    ("e1", "u1", 10.0),
    ("e2", None, 20.0),
    ("e3", "u2", -5.0),
    ("e1", "u1", 10.0),
]

raw = spark.createDataFrame(raw_rows, ["event_id", "user_id", "amount"])
raw.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
|      e2|   NULL|  20.0|
|      e3|     u2|  -5.0|
|      e1|     u1|  10.0|
+--------+-------+------+



## Step 2 — Bronze

Bronze means: store what arrived.

No cleaning.  
No business rules.  
No deduplication.

```text
RAW -> BRONZE
```

In [3]:
bronze = raw

bronze.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
|      e2|   NULL|  20.0|
|      e3|     u2|  -5.0|
|      e1|     u1|  10.0|
+--------+-------+------+



## Step 3 — Silver

Silver means: clean data.

Rules in this simple example:

```text
1. user_id must not be null
2. amount must be >= 0
3. duplicate event_id should be removed
```

So from Bronze we split data into:

```text
BRONZE
  |
  +--> SILVER      = valid clean rows
  |
  +--> QUARANTINE  = invalid rows
```

In [4]:
valid_condition = (
    F.col("user_id").isNotNull()
    & (F.col("amount") >= 0)
)

silver = (
    bronze
    .filter(valid_condition)
    .dropDuplicates(["event_id"])
)

silver.show()

[Stage 4:>                                                          (0 + 2) / 2]

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
+--------+-------+------+



## Step 4 — Quarantine

Quarantine means: keep bad rows separately.

We do not silently lose them.

In [5]:
quarantine = (
    bronze
    .withColumn(
        "error_reason",
        F.when(F.col("user_id").isNull(), F.lit("missing_user_id"))
         .when(F.col("amount") < 0, F.lit("negative_amount"))
    )
    .filter(F.col("error_reason").isNotNull())
)

quarantine.show()

+--------+-------+------+---------------+
|event_id|user_id|amount|   error_reason|
+--------+-------+------+---------------+
|      e2|   NULL|  20.0|missing_user_id|
|      e3|     u2|  -5.0|negative_amount|
+--------+-------+------+---------------+



## Step 5 — Gold

Gold means: business-ready result.

Example:

```text
How much money did each user spend?
```

Gold is usually aggregated, joined, renamed, or modeled for reporting.

In [6]:
gold = (
    silver
    .groupBy("user_id")
    .agg(F.sum("amount").alias("total_amount"))
)

gold.show()

+-------+------------+
|user_id|total_amount|
+-------+------------+
|     u1|        10.0|
+-------+------------+



## Final mental model

```text
BRONZE
- raw
- keep everything
- close to source

SILVER
- cleaned
- valid
- deduplicated

QUARANTINE
- invalid rows
- kept for inspection

GOLD
- business result
- ready for reports / dashboards
```

That is the basic Medallion idea.

In [7]:
spark.stop()